In [8]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 📢 Étape 7 : Data Storytelling & Communication (Squelette Étudiant)

Cette étape correspond au septième et dernier chapitre de data science. L'objectif est de synthétiser les résultats pour les présenter et proposer des visualisations interactives ou dynamiques pour valoriser nos conclusions.

### 1. Préparation de l'environnement

In [16]:
import os
import sys
import pandas as pd
import numpy as np
import datetime

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append(os.path.abspath('..'))

df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

print("Librairies prêtes pour la phase de Data Storytelling !")

Librairies prêtes pour la phase de Data Storytelling !


In [10]:
# On refait le modèle parce qu'on a changé de notebook (on aurait pu le sauvegarder et le recharger, par exemple avec pickle, mais on avait besoin de récupérer les données d'entraînement aussi)

df['Date'] = pd.to_datetime(df['Date'])
min_time = datetime.datetime.timestamp(df['Date'][0])
df['Date'] = df['Date'].map(lambda x : datetime.datetime.timestamp(x) - min_time)

features = ['VTAT', 'CTAT', 'Ride Distance', 'Date']
target = 'Booking Value'

# Préparations des données pour l'entraînement
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

# Entraînement
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# TODO: Calculez MAE, RMSE et R² entre y_test et y_pred
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)


### 2. Visualisation Interactive (Plotly)

On effectue une visualisation Données prédites vs Données réelles pour se donner une idée des performances de notre modèle. On peut constater qu'il a une assez bonne capacité de prédiction dans l'ensemble et on ne détecte que quelques rares cas où il se trompe vraiment.

In [29]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=y_test, y=y_pred,
                         mode='markers', name="Prédiction"))
fig.add_trace(go.Scatter(x=[0,4600], y=[0,4600],
                         mode='lines', name='Prédiction parfaite',
                         marker_color="Black"))
fig.update_layout(title_text="Données réelles vs Données prédites",
                  xaxis_title_text="Données réelles",
                  yaxis_title_text="Données prédites")

fig.show()

### 3. Évolution du modèle en fonction des paramètres

On évalue aussi l'évolution des performances du modèle en fonction du nombre d'estimateurs, que ce soit en termes de précision et de temps de calcul. On peut constater une amélioration constante des performances jusqu'à 30 estimateurs, mais qui devient très légère passés 15 estimateurs (pour un coût en temps qui continue d'évoluer de façon linéaire). Par ailleurs, on peut craindre que des cas de surapprentissage se cachent sous ces dernières bribes d'amélioration. On proposera donc plutôt d'utiliser un modèle utilisant un nombre modéré d'estimateurs, probablement entre 10 et 15, à moins d'avoir de fortes raisons de vouloir prédire nos prix à la roupie près.

In [ ]:
import time

performances = pd.DataFrame(columns=["Estimateurs","Test MAE","Test RMSE","Test R²","Temps"])
for i in range(30) :
    start_time = time.time()
    model = model = RandomForestRegressor(n_estimators=i+1, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    end_time = time.time()
    performances.loc[i] = {"Estimateurs":i+1,"Test MAE":mae,"Test RMSE":rmse,"Test R²":r2,"Temps":end_time-start_time}

In [36]:
performances

,Estimateurs,Test MAE,Test RMSE,Test R²,Temps
0,1,43.077047,69.217315,0.982551,0.463137
1,2,34.045721,55.161846,0.988918,0.824921
2,3,30.506162,50.050153,0.990877,1.254426
3,4,28.515792,47.724670,0.991705,1.581336
4,5,27.249820,45.966587,0.992305,1.915402
5,6,26.380613,44.480935,0.992794,2.280746
6,7,25.623707,43.351149,0.993155,2.689124
7,8,25.122669,42.510369,0.993418,3.057387
8,9,24.742195,42.007511,0.993573,3.487623
9,10,24.373043,41.551451,0.993712,3.813043


In [37]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True)

fig.add_trace(go.Scatter(x=performances["Estimateurs"], y=performances["Test MAE"],
                         mode='lines', name="MAE"),
                         row=1, col=1)
fig.add_trace(go.Scatter(x=performances["Estimateurs"], y=performances["Test RMSE"],
                         mode='lines', name="RMSE"),
                         row=2, col=1)
fig.add_trace(go.Scatter(x=performances["Estimateurs"], y=performances["Test R²"],
                         mode='lines', name="R²"),
                         row=3, col=1)
fig.add_trace(go.Scatter(x=performances["Estimateurs"], y=performances["Temps"],
                         mode='lines', name="Temps"),
                         row=4, col=1)
fig.update_layout(title_text="Évolution des performances du modèle", height=700)

fig.update_xaxes(title_text="Estimateurs",  row=4, col=1)
fig.update_yaxes(title_text="MAE", row=1, col=1)
fig.update_yaxes(title_text="RMSE", row=2, col=1)
fig.update_yaxes(title_text="Score R²", row=3, col=1)
fig.update_yaxes(title_text="Temps (s)", row=4, col=1)

